# Binning & Discretization: Converting Continuous to Categorical

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadtalhaishtiaq/ai-orchestrator/blob/main/03-feature-engineering/05_binning_discretization.ipynb)

## Objectives
- Convert continuous variables to categorical (bins)
- Master equal-width and equal-frequency binning
- Implement domain-specific binning
- Handle edge cases and outliers
- Balance information retention vs model simplification

print("""📊 Benefits of Binning:

✅ Reduce impact of outliers
✅ Create human-interpretable categories
✅ Capture non-linear relationships
✅ Reduce overfitting from continuous noise
✅ Improve model interpretability
✅ Required for some algorithms (e.g., rule-based models)

❌ Drawbacks:
✗ Information loss (smoothing)
✗ May miss subtle patterns
✗ Requires choosing bin count/boundaries
✗ Can create artificial categories""")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

np.random.seed(42)
sns.set_theme()

print("✅ Libraries loaded")

# Create realistic data with outliers
n_samples = 1000
df = pd.DataFrame({
    'age': np.random.normal(35, 15, n_samples),
    'income': np.random.lognormal(10.5, 0.8, n_samples),
    'credit_score': np.random.normal(650, 100, n_samples),
    'spending': np.random.exponential(500, n_samples)
})

# Constrain to realistic ranges
df['age'] = df['age'].clip(18, 100)
df['credit_score'] = df['credit_score'].clip(300, 850)

# Target: high spending if age>40 and income>median
df['high_spender'] = ((df['age'] > 40) & (df['income'] > df['income'].median())).astype(int)

print(f"Dataset shape: {df.shape}")
print(f"\nDescriptive Statistics:")
print(df.describe())

# Equal-width: divide range into equal intervals
df['age_binned_width'] = pd.cut(df['age'], 
                                bins=5,
                                labels=['Very Young', 'Young', 'Middle', 'Senior', 'Elder'])

print("📊 Equal-Width Binning (Age):")
print(f"\nBin boundaries: {pd.cut(df['age'], bins=5).cat.categories}")
print(f"\nBin distribution:")
print(df['age_binned_width'].value_counts().sort_index())

# Equal-frequency: each bin has same number of samples
df['age_binned_freq'] = pd.qcut(df['age'], 
                                q=5,  # 5 quantiles = 5 bins
                                labels=['Q1', 'Q2', 'Q3', 'Q4', 'Q5'],
                                duplicates='drop')

print("📊 Equal-Frequency (Quantile) Binning (Age):")
print(f"\nBin boundaries (quantiles): {pd.qcut(df['age'], q=5, duplicates='drop').cat.categories}")
print(f"\nBin distribution (roughly equal):")
print(df['age_binned_freq'].value_counts().sort_index())

# Define bins based on domain knowledge
age_bins = [0, 25, 35, 50, 65, 100]
age_labels = ['18-25', '26-35', '36-50', '51-65', '65+']
df['age_binned_domain'] = pd.cut(df['age'], bins=age_bins, labels=age_labels, right=False)

print("📊 Domain-Specific Binning (Age):")
print(f"\nBins: {age_bins}")
print(f"Labels: {age_labels}")
print(f"\nBin distribution:")
print(df['age_binned_domain'].value_counts().sort_index())

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Equal-width
axes[0].bar(range(len(df['age_binned_width'].value_counts())), 
            df['age_binned_width'].value_counts().sort_index().values,
            color='blue', alpha=0.7)
axes[0].set_title('Equal-Width Binning\n(Same interval width, unequal counts)')
axes[0].set_ylabel('Count')
axes[0].set_xlabel('Bin')

# Equal-frequency
axes[1].bar(range(len(df['age_binned_freq'].value_counts())), 
            df['age_binned_freq'].value_counts().sort_index().values,
            color='green', alpha=0.7)
axes[1].set_title('Equal-Frequency Binning\n(Same count per bin, unequal widths)')
axes[1].set_ylabel('Count')
axes[1].set_xlabel('Bin')

# Domain-specific
axes[2].bar(range(len(df['age_binned_domain'].value_counts())), 
            df['age_binned_domain'].value_counts().sort_index().values,
            color='orange', alpha=0.7)
axes[2].set_title('Domain-Specific Binning\n(Based on domain knowledge)')
axes[2].set_ylabel('Count')
axes[2].set_xlabel('Bin')

plt.tight_layout()
plt.show()

# Visualize information preservation
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Original continuous
axes[0].hist(df['age'], bins=50, edgecolor='black', alpha=0.7, color='blue')
axes[0].set_title('Original (Continuous)\nFull detail preserved')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')

# Equal-width
bin_counts = df['age_binned_domain'].value_counts().sort_index()
bin_centers = [20, 30, 42.5, 57.5, 82.5]
axes[1].bar(bin_centers, bin_counts.values, width=8, edgecolor='black', alpha=0.7, color='green')
axes[1].set_title('Equal-Width Binned\nInformation loss, but interpretable')
axes[1].set_xlabel('Age')
axes[1].set_ylabel('Count')
axes[1].set_xlim([15, 105])

# Show relationship with target
high_spender_by_age = df.groupby('age_binned_domain')['high_spender'].apply(lambda x: (x==1).sum() / len(x) * 100)
axes[2].bar(range(len(high_spender_by_age)), high_spender_by_age.values, 
            color='orange', alpha=0.7, edgecolor='black')
axes[2].set_xticks(range(len(high_spender_by_age)))
axes[2].set_xticklabels(high_spender_by_age.index, rotation=45)
axes[2].set_title('High Spender % by Age Bin\nBinning captures target relationship')
axes[2].set_ylabel('High Spender %')

plt.tight_layout()
plt.show()

# Apply binning to multiple features
df['income_binned'] = pd.qcut(df['income'], q=4, labels=['Low', 'Medium', 'High', 'Very High'])
df['credit_binned'] = pd.cut(df['credit_score'], 
                              bins=[0, 550, 670, 740, 850],
                              labels=['Poor', 'Fair', 'Good', 'Excellent'])

print("📊 Binned Categorical Features:")
print(f"\nIncome tiers:")
print(df['income_binned'].value_counts().sort_index())
print(f"\nCredit score categories:")
print(df['credit_binned'].value_counts().sort_index())

# Prepare features
# Original continuous
X_continuous = df[['age', 'income', 'credit_score', 'spending']].copy()
y = df['high_spender']

# Binned features
X_binned = df[['age', 'income', 'credit_score', 'spending']].copy()
X_binned['age'] = pd.cut(X_binned['age'], bins=5)
X_binned['income'] = pd.qcut(X_binned['income'], q=4, labels=False, duplicates='drop')
X_binned['credit_score'] = pd.cut(X_binned['credit_score'], bins=4)
X_binned['spending'] = pd.qcut(X_binned['spending'], q=4, labels=False, duplicates='drop')

# Encode categorical features
from sklearn.preprocessing import LabelEncoder
le_dict = {}
for col in ['age', 'credit_score']:
    le = LabelEncoder()
    X_binned[col] = le.fit_transform(X_binned[col].astype(str))
    le_dict[col] = le

# Train-test split
X_train_c, X_test_c, y_train, y_test = train_test_split(X_continuous, y, test_size=0.2, random_state=42)
X_train_b, X_test_b, _, _ = train_test_split(X_binned, y, test_size=0.2, random_state=42)

# Random Forest (handles both well)
rf_c = RandomForestClassifier(n_estimators=100, random_state=42)
rf_c.fit(X_train_c, y_train)
acc_c = accuracy_score(y_test, rf_c.predict(X_test_c))

rf_b = RandomForestClassifier(n_estimators=100, random_state=42)
rf_b.fit(X_train_b, y_train)
acc_b = accuracy_score(y_test, rf_b.predict(X_test_b))

print("📊 Model Performance Comparison:")
print(f"\nContinuous Features: {acc_c:.4f}")
print(f"Binned Features: {acc_b:.4f}")
print(f"\nDifference: {abs(acc_c - acc_b):.4f}")
print(f"\n💡 Both work well! Choose based on interpretability needs.")

# Create data with edge cases
X_edge = pd.DataFrame({
    'value': [1, 5, 10, 15, 20, 25, 30, np.nan, np.inf, -np.inf]
})

print("📊 Edge Cases in Binning:")
print(f"Original data:\n{X_edge}")

# Handle NaN before binning
X_edge_clean = X_edge.dropna()
X_edge_clean = X_edge_clean[np.isfinite(X_edge_clean['value'])]

# Now bin
X_edge_clean['binned'] = pd.cut(X_edge_clean['value'], bins=3, labels=['Low', 'Medium', 'High'])

print(f"\nAfter handling NaN and inf:\n{X_edge_clean}")

print("""📋 HOW TO CHOOSE NUMBER OF BINS:

1. Sturges' Rule (for normal data):
   n_bins = 1 + log₂(n)
   Example: 1000 samples → ~10 bins

2. Rice Rule:
   n_bins = 2 * n^(1/3)
   More bins than Sturges

3. Square Root Rule:
   n_bins = √n
   Simple and popular

4. Domain Knowledge:
   Choose based on business logic
   Example: Age groups (18-25, 26-35, etc.)

5. Cross-Validation:
   Test different bin counts
   Choose best test performance

⚠️  General Guidelines:
   • Too few bins: Information loss
   • Too many bins: Overfitting risk
   • Usually: 3-10 bins work well""")

# Calculate for our data
n_samples = len(df)
sturgess_bins = int(1 + np.log2(n_samples))
square_root_bins = int(np.sqrt(n_samples))

print(f"\n📊 For our dataset ({n_samples} samples):")
print(f"  Sturges' Rule: {sturgess_bins} bins")
print(f"  Square Root Rule: {square_root_bins} bins")

print("""\n📚 KEY TAKEAWAYS:

Binning Methods:
✓ Equal-Width: Simple, unequal sample counts
✓ Equal-Frequency: Balanced samples, easier interpretation
✓ Domain-Specific: Most interpretable, requires domain knowledge

When to Use:
✓ Reduce impact of outliers
✓ Create interpretable categories
✓ For rule-based or decision tree models
✓ When domain has natural breakpoints

Trade-offs:
✓ Information loss vs interpretability
✓ Simpler model vs less accurate

Best Practices:
✓ Use 3-10 bins (usually)
✓ Handle NaN/inf before binning
✓ Validate with domain experts
✓ Compare binned vs continuous on test set

Next: DateTime feature extraction!""")